In [1]:
from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# MVTec dataset path
MVTEC_PATH = PROJECT_ROOT / "datasets" / "mvtec_anomaly_detection"

# Output folder
METADATA_PATH = PROJECT_ROOT / "metadata"
METADATA_PATH.mkdir(exist_ok=True)

print("MVTec path:", MVTEC_PATH)
print("Metadata output path:", METADATA_PATH)

MVTec path: c:\Users\rimjh\IndustrialDefectProject\datasets\mvtec_anomaly_detection
Metadata output path: c:\Users\rimjh\IndustrialDefectProject\metadata


In [2]:
# Generate metadata for all MVTec images

image_extensions = {".png", ".jpg", ".jpeg", ".bmp"}

rows = []

categories = sorted([
    folder.name
    for folder in MVTEC_PATH.iterdir()
    if folder.is_dir()
])

for category in categories:
    category_path = MVTEC_PATH / category

    # -------------------------
    # Training images
    # -------------------------
    train_path = category_path / "train" / "good"

    if train_path.exists():
        for image_path in sorted(train_path.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in image_extensions:
                rows.append({
                    "dataset": "MVTec",
                    "category": category,
                    "defect_type": "good",
                    "label": 0,
                    "image_path": str(image_path),
                    "mask_path": "",
                    "split": "train"
                })

    # -------------------------
    # Test images
    # -------------------------
    test_path = category_path / "test"
    ground_truth_path = category_path / "ground_truth"

    for defect_folder in sorted(test_path.iterdir()):

        if not defect_folder.is_dir():
            continue

        defect_type = defect_folder.name

        for image_path in sorted(defect_folder.iterdir()):

            if not image_path.is_file():
                continue

            if image_path.suffix.lower() not in image_extensions:
                continue

            # Normal image
            if defect_type == "good":
                label = 0
                mask_path = ""

            # Defective image
            else:
                label = 1

                mask_file = (
                    ground_truth_path
                    / defect_type
                    / f"{image_path.stem}_mask{image_path.suffix}"
                )

                mask_path = str(mask_file)

            rows.append({
                "dataset": "MVTec",
                "category": category,
                "defect_type": defect_type,
                "label": label,
                "image_path": str(image_path),
                "mask_path": mask_path,
                "split": "test"
            })

# Convert to DataFrame
mvtec_metadata = pd.DataFrame(rows)

print("Total metadata rows:", len(mvtec_metadata))
print("\nColumns:")
print(mvtec_metadata.columns.tolist())

mvtec_metadata.head()

Total metadata rows: 5354

Columns:
['dataset', 'category', 'defect_type', 'label', 'image_path', 'mask_path', 'split']


,dataset,category,defect_type,label,image_path,mask_path,split
0,MVTec,bottle,good,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,train
1,MVTec,bottle,good,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,train
2,MVTec,bottle,good,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,train
3,MVTec,bottle,good,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,train
4,MVTec,bottle,good,0,c:\Users\rimjh\IndustrialDefectProject\dataset...,,train


In [3]:
print("Label distribution:")
print(mvtec_metadata["label"].value_counts())

print("\nSplit distribution:")
print(mvtec_metadata["split"].value_counts())

print("\nDataset categories:")
print(mvtec_metadata["category"].nunique())

Label distribution:
label
0    4096
1    1258
Name: count, dtype: int64

Split distribution:
split
train    3629
test     1725
Name: count, dtype: int64

Dataset categories:
15


In [4]:
# Check image paths

image_exists = mvtec_metadata["image_path"].apply(
    lambda x: Path(x).exists()
)

print("Images found:", image_exists.sum())
print("Images missing:", (~image_exists).sum())

Images found: 5354
Images missing: 0


In [5]:
# Check masks for defective images

defective_data = mvtec_metadata[mvtec_metadata["label"] == 1]

mask_exists = defective_data["mask_path"].apply(
    lambda x: Path(x).exists()
)

print("Defective images:", len(defective_data))
print("Masks found:", mask_exists.sum())
print("Masks missing:", (~mask_exists).sum())

Defective images: 1258
Masks found: 1258
Masks missing: 0


In [6]:
# Save MVTec metadata

output_file = METADATA_PATH / "mvtec_metadata.csv"

mvtec_metadata.to_csv(output_file, index=False)

print("Metadata saved successfully!")
print("File:", output_file)
print("Rows:", len(mvtec_metadata))

Metadata saved successfully!
File: c:\Users\rimjh\IndustrialDefectProject\metadata\mvtec_metadata.csv
Rows: 5354
